# 03 — Imbalance handling

Fraud represents only about 0.17% of transactions. This notebook compares class weighting, random undersampling, and SMOTE using cross-validation on the training split only. Resampling is never applied to the test set, because synthetic or discarded test observations would invalidate the final evaluation.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from src.data_utils import split_features_target

df = pd.read_csv("../Dataset/creditcard_cleaned.csv")
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
fraud_weight = (y_train == 0).sum() / (y_train == 1).sum()
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


## Candidate approaches

Class weighting changes the loss function without changing data. Undersampling makes training faster but discards many legitimate examples. SMOTE creates synthetic minority examples, so scaling precedes it for the distance calculation. Whether any approach improves PR-AUC is an empirical question, not an assumption.

In [2]:
candidates = {
    'Logistic Regression — class weight': ImbPipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    ]),
    'Random Forest — class weight': RandomForestClassifier(
        n_estimators=200, class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ),
    'XGBoost — imbalance weight': XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1, scale_pos_weight=fraud_weight,
        eval_metric='logloss', random_state=42, n_jobs=-1
    ),
    'Logistic Regression — undersampling': ImbPipeline([
        ('scale', StandardScaler()),
        ('sampler', RandomUnderSampler(random_state=42)),
        ('model', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'Logistic Regression — SMOTE': ImbPipeline([
        ('scale', StandardScaler()),
        ('sampler', SMOTE(random_state=42)),
        ('model', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
}

scoring = {'PR-AUC': 'average_precision', 'ROC-AUC': 'roc_auc', 'F1': 'f1', 'Recall': 'recall', 'Precision': 'precision'}
rows = []
for name, pipeline in candidates.items():
    scores = cross_validate(pipeline, X_train, y_train, scoring=scoring, cv=cv, n_jobs=-1)
    rows.append({'Approach': name, **{metric: scores[f'test_{metric}'].mean() for metric in scoring}})
imbalance_comparison = pd.DataFrame(rows).sort_values('PR-AUC', ascending=False)
display(imbalance_comparison.style.format({column: '{:.4f}' for column in imbalance_comparison.columns[1:]}))


,Approach,PR-AUC,ROC-AUC,F1,Recall,Precision
2,XGBoost — imbalance weight,0.8501,0.9781,0.8381,0.8459,0.8331
1,Random Forest — class weight,0.8399,0.9548,0.8392,0.7544,0.9481
4,Logistic Regression — SMOTE,0.7455,0.9697,0.1103,0.8982,0.0588
0,Logistic Regression — class weight,0.7451,0.9741,0.1166,0.8956,0.0624
3,Logistic Regression — undersampling,0.5876,0.9761,0.0687,0.9139,0.0357


## Conclusion

The imbalance-handling experiments showed that different strategies produced substantially different results across the three machine learning algorithms. Among the approaches evaluated using 3-fold stratified cross-validation, **XGBoost with imbalance weighting achieved the highest mean PR-AUC (0.8501)**, followed closely by **Random Forest with class weighting (0.8399)**. XGBoost also achieved a strong balance between recall (0.8459) and precision (0.8331), indicating that it was able to identify a relatively high proportion of fraudulent transactions while keeping false fraud alerts under control.

Random Forest showed the **highest precision (0.9481)** among the approaches, meaning that its fraud alerts were highly accurate. However, its recall was lower (0.7544) than XGBoost, indicating that it missed more fraudulent transactions. In contrast, the Logistic Regression approaches using SMOTE, class weighting, and undersampling achieved relatively high recall but very low precision, particularly undersampling, which achieved a recall of 0.9139 but a precision of only 0.0357. This demonstrates that maximizing recall alone can result in a very large number of false-positive fraud alerts.

Overall, **XGBoost with imbalance weighting was selected as the strongest candidate for further development**, based primarily on its highest PR-AUC and its more balanced precision-recall performance. However, model selection in fraud detection should not depend solely on a single metric. A key business question is:

> **Is it more costly for the business to miss a fraudulent transaction, or to investigate a legitimate transaction that has been incorrectly flagged as fraud?**

If missing fraud is considered substantially more costly, a higher-recall model may be preferred even if it produces more false positives. Conversely, if investigating false alarms is expensive or negatively affects legitimate customers, a higher-precision model such as Random Forest may be preferable.

Therefore, the final model should ultimately be selected by considering both **statistical performance and the practical cost of false negatives versus false positives**. The next stage of the project will focus on **hyperparameter tuning**, particularly for XGBoost and Random Forest, to determine whether their performance can be further improved.
